# Explainable CNN for Potato Leaf Disease Detection

This notebook trains and evaluates the lightweight CNN used by the application. It is intentionally focused on one reproducible pipeline rather than several competing experiments.

The model classifies five potato leaf categories: Bacteria, Fungi, Healthy, Phytopthora, and Virus. The preprocessing, class order, model input shape, and exported model artifact are kept compatible with the FastAPI inference service.

## Pipeline

1. Load and audit the dataset.
2. Create a leak-aware, stratified train/validation/test split.
3. Apply training augmentation and shared evaluation preprocessing.
4. Train the model (MobileNetV2 transfer learning by default) with macro-F1 checkpointing and early stopping.
5. Evaluate with accuracy, a classification report, and a confusion matrix.
6. Generate a prediction smoke test and Grad-CAM examples.
7. Export the model and the metadata required to serve it safely.

> Dataset and deployment limitations are recorded in the final section.


In [ ]:
# Imports and reproducibility
import json
import os
import random
from pathlib import Path

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_DETERMINISTIC_OPS', '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from PIL import Image as PILImage
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError):
    pass

# Avoid a TensorFlow Grappler layout-optimizer issue with deterministic GPU dropout.
tf.config.optimizer.set_experimental_options({'layout_optimizer': False})

GPU_DEVICES = tf.config.list_physical_devices('GPU')
for gpu in GPU_DEVICES:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
strategy = tf.distribute.MirroredStrategy()
GPU_NAMES = [device.name for device in GPU_DEVICES]

tf.get_logger().setLevel('ERROR')
sns.set_theme(style='whitegrid')
print(f'TensorFlow: {tf.__version__}')
print(f'Keras: {keras.__version__}')
print(f'Visible GPUs: {GPU_NAMES}')
print(f'Replicas in sync: {strategy.num_replicas_in_sync}')


## 1. Configuration and Dataset Contract

Set `DATASET_DIR` to the directory that contains the five potato class folders (`Bacteria`, `Fungi`, `Healthy`, `Phytopthora`, `Virus`).

The explicit `CLASS_NAMES` order is part of the model contract. It must not be inferred from filesystem ordering.


In [ ]:
# Paths and model contract
IMG_SIZE = (224, 224)
CHANNELS = 3
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '32'))
EPOCHS = int(os.getenv('EPOCHS', '60'))
LEARNING_RATE = float(os.getenv('LEARNING_RATE', '3e-4'))
CLASS_WEIGHT_CAP = float(os.getenv('CLASS_WEIGHT_CAP', '3.0'))
MIN_MACRO_F1_MARGIN = float(os.getenv('MIN_MACRO_F1_MARGIN', '0.10'))
SANITY_MIN_MEAN_CONFIDENCE = float(os.getenv('SANITY_MIN_MEAN_CONFIDENCE', '0.45'))
ARCHITECTURE = os.getenv('ARCHITECTURE', 'mobilenetv2').lower()
HEAD_LEARNING_RATE = float(os.getenv('HEAD_LEARNING_RATE', '1e-3'))
FINE_TUNE_LEARNING_RATE = float(os.getenv('FINE_TUNE_LEARNING_RATE', '1e-4'))
FINE_TUNE_AT_EPOCH = int(os.getenv('FINE_TUNE_AT_EPOCH', '10'))
FINE_TUNE_LAYERS = int(os.getenv('FINE_TUNE_LAYERS', '54'))
WEIGHT_DECAY = float(os.getenv('WEIGHT_DECAY', '1e-4'))
LABEL_SMOOTHING = float(os.getenv('LABEL_SMOOTHING', '0.05'))
DROPOUT_RATE = float(os.getenv('DROPOUT_RATE', '0.25'))
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

CLASS_NAMES = [
    'Bacteria',
    'Fungi',
    'Healthy',
    'Phytopthora',
    'Virus',
]
DISPLAY_NAMES = {
    'Bacteria': 'Bacterial Disease',
    'Fungi': 'Fungal Disease',
    'Healthy': 'Healthy',
    'Phytopthora': 'Phytophthora',
    'Virus': 'Viral Disease',
}
EXPECTED_CLASS_INDICES = {name: index for index, name in enumerate(CLASS_NAMES)}
DATASET_DIR = Path(os.getenv('DATASET_DIR', '/kaggle/input/datasets/mamunonweb/potato-disease-dataset-03')).expanduser()
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', 'artifacts')).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep the best checkpoint and the final-epoch model separate so they can be
# compared before a single model is exported for serving.
CHECKPOINT_PATH = OUTPUT_DIR / 'checkpoint_best.keras'
FINE_TUNE_CHECKPOINT_PATH = OUTPUT_DIR / 'checkpoint_fine_tune.keras'
FINAL_MODEL_PATH = OUTPUT_DIR / 'final_model.keras'
DEPLOY_MODEL_PATH = OUTPUT_DIR / 'explainable_cnn_model.keras'

print(f'Dataset: {DATASET_DIR.resolve()}')
print(f'Output:  {OUTPUT_DIR.resolve()}')
print(f'Classes: {CLASS_NAMES}')
print(f'Input:   {IMG_SIZE + (CHANNELS,)}')
print(f'Architecture: {ARCHITECTURE} | Class weight cap: {CLASS_WEIGHT_CAP}')
print(f'Head LR: {HEAD_LEARNING_RATE} | Fine-tune LR: {FINE_TUNE_LEARNING_RATE} | Weight decay: {WEIGHT_DECAY}')
print(f'Epochs: {EPOCHS} (head {FINE_TUNE_AT_EPOCH}) | Fine-tune top {FINE_TUNE_LAYERS} layers (BN frozen) | Label smoothing: {LABEL_SMOOTHING}')


In [ ]:
def build_image_dataframe(dataset_dir):
    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f'Dataset directory does not exist: {dataset_dir}. '
            'Set DATASET_DIR to the directory containing the class folders.'
        )

    rows = []
    ignored_classes = []
    for class_dir in sorted(path for path in dataset_dir.iterdir() if path.is_dir()):
        if class_dir.name not in CLASS_NAMES:
            ignored_classes.append(class_dir.name)
            continue
        for image_path in sorted(class_dir.rglob('*')):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({'filepath': str(image_path.resolve()), 'label': class_dir.name})

    data = pd.DataFrame(rows, columns=['filepath', 'label'])
    if data.empty:
        raise ValueError(
            f'No supported images found in {dataset_dir}. '
            f'Expected class folders: {CLASS_NAMES}'
        )

    missing_classes = sorted(set(CLASS_NAMES) - set(data['label']))
    if missing_classes:
        raise ValueError(f'Missing expected class folders: {missing_classes}')
    if ignored_classes:
        print(f'Ignoring non-potato class folders: {ignored_classes}')

    return data.sort_values(['label', 'filepath']).reset_index(drop=True)


data = build_image_dataframe(DATASET_DIR)
counts = data['label'].value_counts().reindex(CLASS_NAMES)
print(f'Total images: {len(data):,}')
display(counts.rename('images').to_frame())

## 2. Dataset Audit

This is a compact audit: class balance and one representative image per class. The detailed RGB histogram exploration from the original notebook was removed because it did not affect preprocessing, architecture, or evaluation.

In [ ]:
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(18, 4))
for axis, class_name in zip(axes, CLASS_NAMES):
    sample = data[data['label'] == class_name].sample(1, random_state=SEED).iloc[0]
    with PILImage.open(sample['filepath']) as image:
        axis.imshow(image.convert('RGB'))
    axis.set_title(f"{DISPLAY_NAMES[class_name]}\n({counts[class_name]:,} images)")
    axis.axis('off')
fig.suptitle('Representative samples by class')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'dataset_samples.png', dpi=160, bbox_inches='tight')
plt.show()

## 3. Stratified Train, Validation, and Test Split

The split is 80/10/10. Stratification preserves class proportions in every set, while the filepath checks prevent accidental overlap between sets.

In [ ]:
DUPLICATE_HASH_THRESHOLD = int(os.getenv('DUPLICATE_HASH_THRESHOLD', '10'))


def difference_hash(filepath, hash_size=8):
    """Perceptual (difference) hash used to detect near-duplicate images."""
    with PILImage.open(filepath) as image:
        resized = image.convert('L').resize(
            (hash_size + 1, hash_size), PILImage.Resampling.LANCZOS
        )
    pixels = np.asarray(resized, dtype=np.int16)
    bits = (pixels[:, 1:] > pixels[:, :-1]).flatten()
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return value


def compute_duplicate_groups(dataframe, hash_threshold):
    """Group near-duplicate images so they never straddle train/val/test.

    Bursts, re-crops, and copies of the same leaf leak information across a
    random split and inflate validation metrics. Grouping by perceptual hash
    keeps each duplicate family inside a single split.
    """
    hashes = np.array(
        [difference_hash(path) for path in dataframe['filepath']], dtype=np.uint64
    )
    count = len(hashes)
    parent = list(range(count))

    def find(index):
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    def union(left, right):
        left_root, right_root = find(left), find(right)
        if left_root != right_root:
            parent[right_root] = left_root

    def popcount(values):
        values = values - ((values >> np.uint64(1)) & np.uint64(0x5555555555555555))
        values = (values & np.uint64(0x3333333333333333)) + (
            (values >> np.uint64(2)) & np.uint64(0x3333333333333333)
        )
        values = (values + (values >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F)
        values = values + (values >> np.uint64(8))
        values = values + (values >> np.uint64(16))
        values = values + (values >> np.uint64(32))
        return (values & np.uint64(0x7F)).astype(np.int64)

    block = 400
    for start in range(0, count, block):
        distances = popcount(hashes[start:start + block, None] ^ hashes[None, :])
        rows, columns = np.where(distances <= hash_threshold)
        for row, column in zip(rows, columns):
            left, right = start + int(row), int(column)
            if left < right:
                union(left, right)

    roots = np.array([find(index) for index in range(count)])
    _, group_ids = np.unique(roots, return_inverse=True)
    return group_ids


data = data.reset_index(drop=True)
data['group'] = compute_duplicate_groups(data, DUPLICATE_HASH_THRESHOLD)
print(
    f'Duplicate grouping: {data["group"].nunique():,} groups from '
    f'{len(data):,} images (dHash <= {DUPLICATE_HASH_THRESHOLD}).'
)

test_splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
train_index, test_index = next(
    test_splitter.split(data, data['label'], groups=data['group'])
)
remaining_data = data.iloc[train_index].reset_index(drop=True)

validation_splitter = StratifiedGroupKFold(
    n_splits=9, shuffle=True, random_state=SEED
)
train_index, validation_index = next(
    validation_splitter.split(
        remaining_data, remaining_data['label'], groups=remaining_data['group']
    )
)
train_data = remaining_data.iloc[train_index].reset_index(drop=True)
valid_data = remaining_data.iloc[validation_index].reset_index(drop=True)
test_data = data.iloc[test_index].reset_index(drop=True)

train_groups = set(train_data['group'])
valid_groups = set(valid_data['group'])
test_groups = set(test_data['group'])
assert train_groups.isdisjoint(valid_groups)
assert train_groups.isdisjoint(test_groups)
assert valid_groups.isdisjoint(test_groups)

split_summary = pd.DataFrame(
    {
        'train': train_data['label'].value_counts().reindex(CLASS_NAMES),
        'validation': valid_data['label'].value_counts().reindex(CLASS_NAMES),
        'test': test_data['label'].value_counts().reindex(CLASS_NAMES),
    }
).fillna(0).astype(int)
display(split_summary)
print(f'Train: {len(train_data):,} | Validation: {len(valid_data):,} | Test: {len(test_data):,}')


## 4. Preprocessing and Data Generators

Images are resized to `224x224`. The generators scale pixels from `[0, 255]` to `[0, 1]`; the MobileNetV2 model then rescales `[0, 1]` to `[-1, 1]` to match its pretrained input. Training augmentation is applied only to the training set. Validation and test images use rescaling only.

The explicit `classes` argument guarantees that generator indices match the FastAPI class order.


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    zoom_range=0.20,
    brightness_range=(0.85, 1.15),
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest',
)
evaluation_datagen = ImageDataGenerator(rescale=1.0 / 255.0)
if BATCH_SIZE % strategy.num_replicas_in_sync != 0:
    raise ValueError(
        'BATCH_SIZE must be divisible by the number of synchronized replicas.'
    )

generator_options = {
    'x_col': 'filepath',
    'y_col': 'label',
    'classes': CLASS_NAMES,
    'target_size': IMG_SIZE,
    'color_mode': 'rgb',
    'class_mode': 'categorical',
    'batch_size': BATCH_SIZE,
    'validate_filenames': True,
}

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_data, shuffle=True, seed=SEED, **generator_options
)
valid_gen = evaluation_datagen.flow_from_dataframe(
    dataframe=valid_data, shuffle=False, **generator_options
)
test_gen = evaluation_datagen.flow_from_dataframe(
    dataframe=test_data, shuffle=False, **generator_options
)

assert train_gen.class_indices == EXPECTED_CLASS_INDICES
batch_images, batch_labels = train_gen[0]
assert batch_images.shape[1:] == IMG_SIZE + (CHANNELS,)
assert batch_labels.shape[1] == len(CLASS_NAMES)
assert float(batch_images.min()) >= -1e-6 and float(batch_images.max()) <= 1.0 + 1e-6
print(f'Batch images: {batch_images.shape}')
print(f'Batch labels: {batch_labels.shape}')
print(f'Class mapping: {train_gen.class_indices}')

## 5. Model Architecture

The default model is **MobileNetV2** pretrained on ImageNet. The convolutional base is frozen while a new classification head is trained, and its top layers are later unfrozen for fine-tuning. Set `ARCHITECTURE=custom_cnn` to train the original lightweight custom CNN instead. The final layer expands to the five dataset classes.


In [ ]:
def build_custom_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape, name='image')

    x = layers.Conv2D(32, 3, padding='same', name='conv_1')(inputs)
    x = layers.BatchNormalization(name='batch_norm_1')(x)
    x = layers.Activation('swish', name='swish_1')(x)
    x = layers.Conv2D(64, 3, padding='same', name='conv_2')(x)
    x = layers.BatchNormalization(name='batch_norm_2')(x)
    x = layers.Activation('swish', name='swish_2')(x)
    x = layers.MaxPooling2D(2, name='pool_1')(x)
    x = layers.Dropout(0.25, name='dropout_1')(x)

    x = layers.Conv2D(64, 3, padding='same', name='conv_3')(x)
    x = layers.BatchNormalization(name='batch_norm_3')(x)
    x = layers.Activation('swish', name='swish_3')(x)
    x = layers.Conv2D(128, 3, padding='same', name='conv_4')(x)
    x = layers.BatchNormalization(name='batch_norm_4')(x)
    x = layers.Activation('swish', name='swish_4')(x)
    x = layers.MaxPooling2D(2, name='pool_2')(x)
    x = layers.Dropout(0.25, name='dropout_2')(x)

    x = layers.Conv2D(128, 3, padding='same', name='conv_5')(x)
    x = layers.BatchNormalization(name='batch_norm_5')(x)
    x = layers.Activation('swish', name='swish_5')(x)
    x = layers.Conv2D(256, 3, padding='same', name='conv_6')(x)
    x = layers.BatchNormalization(name='batch_norm_6')(x)
    x = layers.Activation('swish', name='swish_6')(x)
    x = layers.MaxPooling2D(2, name='pool_3')(x)
    x = layers.Dropout(0.30, name='dropout_3')(x)

    x = layers.GlobalAveragePooling2D(name='global_average_pool')(x)
    x = layers.Dense(256, name='dense_features')(x)
    x = layers.BatchNormalization(name='batch_norm_dense')(x)
    x = layers.Activation('swish', name='swish_dense')(x)
    x = layers.Dropout(0.50, name='dropout_dense')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='classifier')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='potato_disease_cnn')
    return model, None


def build_mobilenetv2(input_shape, num_classes):
    import inspect

    inputs = keras.Input(shape=input_shape, name='image')
    # The generators feed pixels in [0, 1]; MobileNetV2 expects [-1, 1].
    x = layers.Rescaling(scale=2.0, offset=-1.0, name='mobilenetv2_preprocessing')(inputs)
    backbone_kwargs = {
        'input_shape': input_shape,
        'include_top': False,
        'weights': 'imagenet',
    }
    # Keras 3 can add its own preprocessing; disable it so our rescaling is the
    # only normalization. Older Keras versions have no such argument and expect
    # the caller to preprocess, which our rescaling already does.
    if 'include_preprocessing' in inspect.signature(
        keras.applications.MobileNetV2
    ).parameters:
        backbone_kwargs['include_preprocessing'] = False
    backbone = keras.applications.MobileNetV2(**backbone_kwargs)
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name='global_average_pool')(x)
    x = layers.Dropout(DROPOUT_RATE, name='dropout_features')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='classifier')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='potato_mobilenetv2')
    return model, backbone


def build_model(input_shape, num_classes, architecture):
    architecture = architecture.lower()
    if architecture == 'mobilenetv2':
        return build_mobilenetv2(input_shape, num_classes)
    if architecture == 'custom_cnn':
        return build_custom_cnn(input_shape, num_classes)
    raise ValueError(
        f"Unknown ARCHITECTURE '{architecture}'. Use 'mobilenetv2' or 'custom_cnn'."
    )


with strategy.scope():
    model, backbone = build_model(IMG_SIZE + (CHANNELS,), len(CLASS_NAMES), ARCHITECTURE)
model.summary()
print(f'Architecture: {ARCHITECTURE}')
if backbone is not None:
    print(f'Backbone layers: {len(backbone.layers)} (frozen during head training)')



## 6. Compile and Train

Label smoothing reduces overconfident predictions, and class weights computed from the training split are capped so the smallest class cannot dominate the loss. Checkpoints and early stopping monitor validation **macro-F1** rather than validation loss, so a model that merely predicts the majority class is never selected. For MobileNetV2, the backbone is unfrozen partway through training and recompiled with a smaller learning rate.


In [ ]:
train_class_counts = train_data['label'].value_counts().reindex(CLASS_NAMES)
raw_class_weights = {
    class_index: float(
        len(train_data) / (len(CLASS_NAMES) * int(class_count))
    )
    for class_index, class_count in enumerate(train_class_counts)
}
# Cap the inverse-frequency weights so the rarest class cannot dominate the
# loss, then rescale to a mean of 1.0 to keep the effective learning rate stable.
CLASS_WEIGHTS = {
    class_index: min(weight, CLASS_WEIGHT_CAP)
    for class_index, weight in raw_class_weights.items()
}
mean_class_weight = float(np.mean(list(CLASS_WEIGHTS.values())))
CLASS_WEIGHTS = {
    class_index: weight / mean_class_weight
    for class_index, weight in CLASS_WEIGHTS.items()
}
print(f'Train class counts: {train_class_counts.to_dict()}')
print(f'Raw class weights: {raw_class_weights}')
print(f'Capped class weights: {CLASS_WEIGHTS}')


class MacroF1Callback(keras.callbacks.Callback):
    """Add validation macro-F1 to the epoch logs.

    Selecting on macro-F1 (instead of validation loss) prevents a checkpoint
    that merely predicts the majority class from being treated as the best model.
    """

    def __init__(self, generator, prefix='val'):
        super().__init__()
        self.generator = generator
        self.prefix = prefix

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        self.generator.reset()
        y_true = np.asarray(self.generator.classes)
        probabilities = self.model.predict(self.generator, verbose=0)
        self.generator.reset()
        y_pred = np.argmax(probabilities, axis=1)
        logs[f'{self.prefix}_macro_f1'] = float(
            f1_score(y_true, y_pred, average='macro', zero_division=0)
        )


def make_callbacks(checkpoint_path, patience, lr_patience=4):
    return [
        MacroF1Callback(valid_gen, prefix='val'),
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_macro_f1',
            mode='max',
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_macro_f1',
            mode='max',
            patience=patience,
            restore_best_weights=False,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_macro_f1',
            mode='max',
            factor=0.5,
            patience=lr_patience,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


def combine_histories(histories):
    # Only keep metrics logged in every segment so the columns stay aligned.
    shared_keys = set(histories[0].history)
    for history in histories[1:]:
        shared_keys &= set(history.history)
    combined = {key: [] for key in shared_keys}
    for history in histories:
        for key in shared_keys:
            combined[key].extend(history.history[key])
    return combined


base_learning_rate = HEAD_LEARNING_RATE if ARCHITECTURE == 'mobilenetv2' else LEARNING_RATE
with strategy.scope():
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=base_learning_rate,
            weight_decay=WEIGHT_DECAY,
            clipnorm=1.0,
        ),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=[keras.metrics.CategoricalAccuracy(name='accuracy')],
    )

history_segments = []
CHECKPOINTS = []

# Transfer learning runs as two separate fit() calls: a frozen-backbone head
# phase, then a fine-tuning phase with the top of the backbone unfrozen while
# keeping all BatchNormalization layers frozen to prevent stat/scale shock.
# Recompiling inside a callback breaks training under MirroredStrategy, so the
# phases are kept as distinct fit() calls instead.
if ARCHITECTURE == 'mobilenetv2' and EPOCHS > FINE_TUNE_AT_EPOCH:
    stage1_history = model.fit(
        train_gen,
        epochs=FINE_TUNE_AT_EPOCH,
        validation_data=valid_gen,
        callbacks=make_callbacks(CHECKPOINT_PATH, patience=6, lr_patience=3),
        class_weight=CLASS_WEIGHTS,
    )
    history_segments.append(stage1_history)
    CHECKPOINTS.append(('head-checkpoint', CHECKPOINT_PATH))

    # Start Stage 2 from the best Stage 1 head checkpoint rather than a degraded last epoch.
    if CHECKPOINT_PATH.exists():
        best_head_model = keras.models.load_model(CHECKPOINT_PATH)
        head_weights = best_head_model.get_weights()
        del best_head_model
        with strategy.scope():
            model.set_weights(head_weights)
        print('Restored best Stage 1 head weights before fine-tuning.')

    backbone.trainable = True
    for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    unfrozen_layers = sum(1 for layer in backbone.layers if layer.trainable)

    with strategy.scope():
        model.compile(
            optimizer=keras.optimizers.AdamW(
                learning_rate=FINE_TUNE_LEARNING_RATE,
                weight_decay=WEIGHT_DECAY,
                clipnorm=1.0,
            ),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
            metrics=[keras.metrics.CategoricalAccuracy(name='accuracy')],
        )
    print(
        f'Fine-tuning: unfroze {unfrozen_layers} non-BN layers from the top '
        f'{FINE_TUNE_LAYERS} backbone layers (lr={FINE_TUNE_LEARNING_RATE}, '
        f'weight_decay={WEIGHT_DECAY}).'
    )
    stage2_start = len(stage1_history.history['loss'])
    history_segments.append(
        model.fit(
            train_gen,
            epochs=EPOCHS,
            initial_epoch=stage2_start,
            validation_data=valid_gen,
            callbacks=make_callbacks(FINE_TUNE_CHECKPOINT_PATH, patience=12, lr_patience=4),
            class_weight=CLASS_WEIGHTS,
        )
    )
    CHECKPOINTS.append(('fine-tune-checkpoint', FINE_TUNE_CHECKPOINT_PATH))
else:
    history_segments.append(
        model.fit(
            train_gen,
            epochs=EPOCHS,
            validation_data=valid_gen,
            callbacks=make_callbacks(CHECKPOINT_PATH, patience=12, lr_patience=4),
            class_weight=CLASS_WEIGHTS,
        )
    )
    CHECKPOINTS.append(('best-checkpoint', CHECKPOINT_PATH))

combined_history = combine_histories(history_segments)
print(
    f"Training segments: {len(history_segments)} | "
    f"epochs: {len(combined_history['loss'])}"
)

# Keep the last-epoch model so it can be compared against the best checkpoints.
final_model = model



## 7. Training History

In [ ]:
history_frame = pd.DataFrame(combined_history)
epochs_completed = range(1, len(history_frame) + 1)
fig, axes = plt.subplots(1, 3, figsize=(19, 4))

axes[0].plot(epochs_completed, history_frame['accuracy'], label='Train')
axes[0].plot(epochs_completed, history_frame['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_completed, history_frame['loss'], label='Train')
axes[1].plot(epochs_completed, history_frame['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

if 'val_macro_f1' in history_frame:
    axes[2].plot(epochs_completed, history_frame['val_macro_f1'], label='Validation')
    axes[2].axhline(1.0 / len(CLASS_NAMES), color='gray', linestyle='--', label='Chance')
    axes[2].set_title('Macro F1 (selection metric)')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()

fig.suptitle('Model training history')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

print(history_frame.tail())


## 8. Test Evaluation

The test set is evaluated only after selecting the best validation checkpoint. The confusion matrix and per-class metrics reveal errors that a single accuracy value can hide.

In [ ]:
final_model.save(FINAL_MODEL_PATH)


def evaluate_model(classifier, generator, name):
    generator.reset()
    metrics = classifier.evaluate(generator, return_dict=True, verbose=0)
    generator.reset()
    probabilities = classifier.predict(generator, verbose=0)
    generator.reset()
    predicted_indices = np.argmax(probabilities, axis=1)
    true_indices = np.asarray(generator.classes)
    macro_f1 = float(
        f1_score(true_indices, predicted_indices, average='macro', zero_division=0)
    )
    report = classification_report(
        true_indices,
        predicted_indices,
        labels=range(len(CLASS_NAMES)),
        target_names=[DISPLAY_NAMES[name] for name in CLASS_NAMES],
        output_dict=True,
        zero_division=0,
    )
    print(
        f'[{name}] loss: {metrics["loss"]:.4f} | '
        f'accuracy: {metrics["accuracy"] * 100:.2f}% | macro-F1: {macro_f1:.4f}'
    )
    return {
        'name': name,
        'model': classifier,
        'test_loss': float(metrics['loss']),
        'test_accuracy': float(metrics['accuracy']),
        'macro_f1': macro_f1,
        'probabilities': probabilities,
        'predicted_indices': predicted_indices,
        'true_indices': true_indices,
        'classification_report': report,
    }


candidate_models = [
    (name, keras.models.load_model(path)) for name, path in CHECKPOINTS
]
candidate_models.append(('final-epoch', keras.models.load_model(FINAL_MODEL_PATH)))

candidate_results = [
    evaluate_model(classifier, test_gen, name)
    for name, classifier in candidate_models
]
selected_result = max(candidate_results, key=lambda result: result['macro_f1'])
selected_name = selected_result['name']
print(f"Selected model: {selected_name} (macro-F1 {selected_result['macro_f1']:.4f})")

# Sanity gate: refuse to export a model that collapsed to a single class,
# cannot beat chance, or is systematically under-confident.
chance_macro_f1 = 1.0 / len(CLASS_NAMES)
unique_predicted = int(len(np.unique(selected_result['predicted_indices'])))
mean_confidence = float(np.mean(np.max(selected_result['probabilities'], axis=1)))
SANITY_CHECKS = {
    'predicts_more_than_one_class': unique_predicted >= 2,
    'macro_f1_above_chance': (
        selected_result['macro_f1'] > chance_macro_f1 + MIN_MACRO_F1_MARGIN
    ),
    'mean_confidence_ok': mean_confidence >= SANITY_MIN_MEAN_CONFIDENCE,
}
print(
    f"Sanity: unique classes={unique_predicted} | "
    f"macro-F1={selected_result['macro_f1']:.4f} (chance {chance_macro_f1:.4f}) | "
    f"mean confidence={mean_confidence:.4f}"
)
for check_name, passed in SANITY_CHECKS.items():
    print(f"  {'PASS' if passed else 'FAIL'}: {check_name}")
if not all(SANITY_CHECKS.values()):
    raise ValueError(
        'Model sanity check failed: the trained model collapsed to the majority '
        'class or is too uncertain to export. Retrain before deploying it.'
    )

selected_model = selected_result['model']
selected_model.save(DEPLOY_MODEL_PATH)
best_model = selected_model
print(f'Exported deployable model: {DEPLOY_MODEL_PATH}')

# Reload the exported artifact and confirm it reproduces the selected model,
# so a corrupted or mismatched .keras file cannot be shipped silently.
reloaded_model = keras.models.load_model(DEPLOY_MODEL_PATH)
test_gen.reset()
reloaded_probabilities = reloaded_model.predict(test_gen, verbose=0)
test_gen.reset()
max_probability_difference = float(
    np.max(np.abs(reloaded_probabilities - selected_result['probabilities']))
)
if max_probability_difference > 1e-3:
    raise ValueError(
        'Exported model does not reproduce the selected model outputs; the '
        '.keras artifact may be corrupted or mismatched.'
    )
print(f'Export integrity check: max probability difference {max_probability_difference:.2e}')

print('Classification report (selected model):')
print(
    classification_report(
        selected_result['true_indices'],
        selected_result['predicted_indices'],
        labels=range(len(CLASS_NAMES)),
        target_names=[DISPLAY_NAMES[name] for name in CLASS_NAMES],
        zero_division=0,
    )
)

matrix = confusion_matrix(
    selected_result['true_indices'],
    selected_result['predicted_indices'],
    labels=range(len(CLASS_NAMES)),
)
fig, axis = plt.subplots(figsize=(7, 6))
sns.heatmap(
    matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=[DISPLAY_NAMES[name] for name in CLASS_NAMES],
    yticklabels=[DISPLAY_NAMES[name] for name in CLASS_NAMES],
    ax=axis,
)
axis.set_xlabel('Predicted label')
axis.set_ylabel('True label')
axis.set_title(f'{ARCHITECTURE} confusion matrix ({selected_name})')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()

# Optional 4-view Test-Time Augmentation (zero extra model params):
# averages predictions over original, horizontal flip, vertical flip, and both flips.
test_gen.reset()
test_batches = [test_gen[batch_idx][0] for batch_idx in range(len(test_gen))]
test_images_array = np.concatenate(test_batches, axis=0)
test_gen.reset()
tta_views = [
    test_images_array,
    test_images_array[:, :, ::-1, :],
    test_images_array[:, ::-1, :, :],
    test_images_array[:, ::-1, ::-1, :],
]
tta_probabilities = np.mean(
    [reloaded_model.predict(view, batch_size=BATCH_SIZE, verbose=0) for view in tta_views],
    axis=0,
)
tta_predicted_indices = np.argmax(tta_probabilities, axis=1)
tta_true_indices = selected_result['true_indices']
tta_accuracy = float(np.mean(tta_predicted_indices == tta_true_indices))
tta_macro_f1 = float(
    f1_score(tta_true_indices, tta_predicted_indices, average='macro', zero_division=0)
)
tta_report = classification_report(
    tta_true_indices,
    tta_predicted_indices,
    labels=range(len(CLASS_NAMES)),
    target_names=[DISPLAY_NAMES[name] for name in CLASS_NAMES],
    output_dict=True,
    zero_division=0,
)
print(
    f'[4-view TTA] accuracy: {tta_accuracy * 100:.2f}% | '
    f'macro-F1: {tta_macro_f1:.4f} (same {DEPLOY_MODEL_PATH.name})'
)
TTA_METRICS = {
    'views': 4,
    'test_accuracy': tta_accuracy,
    'test_macro_f1': tta_macro_f1,
    'classification_report': tta_report,
}

evaluation_artifact = {
    'selected_model': selected_name,
    'test_loss': selected_result['test_loss'],
    'test_accuracy': selected_result['test_accuracy'],
    'test_macro_f1': selected_result['macro_f1'],
    'tta_metrics': TTA_METRICS,
    'chance_macro_f1': chance_macro_f1,
    'test_samples': int(len(test_data)),
    'mean_confidence': mean_confidence,
    'unique_predicted_classes': unique_predicted,
    'sanity_checks': SANITY_CHECKS,
    'classification_report': selected_result['classification_report'],
    'candidates': [
        {
            'name': result['name'],
            'test_loss': result['test_loss'],
            'test_accuracy': result['test_accuracy'],
            'macro_f1': result['macro_f1'],
        }
        for result in candidate_results
    ],
}
(OUTPUT_DIR / 'evaluation.json').write_text(json.dumps(evaluation_artifact, indent=2))


## 9. Inference Smoke Test

This uses the same resize and `/255` preprocessing as the backend. It is a quick contract check before generating explanations or handing the artifact to the application.

In [ ]:
def prepare_image(filepath):
    image = keras.utils.load_img(filepath, target_size=IMG_SIZE, color_mode='rgb')
    array = keras.utils.img_to_array(image).astype('float32') / 255.0
    return image, np.expand_dims(array, axis=0)


def predict_filepath(filepath, classifier):
    image, batch = prepare_image(filepath)
    probabilities = classifier(batch, training=False).numpy()[0]
    predicted_index = int(np.argmax(probabilities))
    return image, predicted_index, probabilities


sample_row = test_data.sample(1, random_state=SEED).iloc[0]
sample_image, sample_index, sample_probabilities = predict_filepath(
    sample_row['filepath'], best_model
)
print(f"True:      {DISPLAY_NAMES[sample_row['label']]}")
print(f"Predicted: {DISPLAY_NAMES[CLASS_NAMES[sample_index]]}")
print(f"Confidence: {sample_probabilities[sample_index] * 100:.2f}%")
assert len(sample_probabilities) == len(CLASS_NAMES)

plt.figure(figsize=(5, 5))
plt.imshow(sample_image)
plt.title(
    f"True: {DISPLAY_NAMES[sample_row['label']]} | "
    f"Predicted: {DISPLAY_NAMES[CLASS_NAMES[sample_index]]}"
)
plt.axis('off')
plt.show()

## 10. Grad-CAM Explainability

Grad-CAM visualizes the regions that contributed to a selected class score. It is an attention explanation, not a segmentation mask and not a validated percentage of infected tissue. The final convolutional layer is selected from the same model used for prediction, so a second architecture definition is unnecessary.

In [ ]:
def find_last_conv_layer(classifier):
    # Transfer-learning backbones are nested models; search inside them first.
    for layer in classifier.layers:
        if isinstance(layer, keras.Model):
            for candidate in reversed(layer.layers):
                if isinstance(candidate, layers.Conv2D):
                    return candidate
    for layer in reversed(classifier.layers):
        if isinstance(layer, layers.Conv2D):
            return layer
    raise ValueError('No Conv2D layer found.')


def _apply_layers(layer_list, tensor):
    for layer in layer_list:
        if isinstance(layer, layers.InputLayer):
            continue
        tensor = layer(tensor, training=False)
    return tensor


def make_gradcam_heatmap(image_batch, classifier, class_index=None, target_layer=None):
    # Nested transfer-learning backbones cannot be re-traced layer by layer, so
    # use the backbone's own graph to tap the last convolutional feature map and
    # apply the surrounding preprocessing/head layers around it.
    backbone = next(
        (layer for layer in classifier.layers if isinstance(layer, keras.Model)),
        None,
    )

    with tf.GradientTape() as tape:
        if backbone is None:
            target_layer = target_layer or find_last_conv_layer(classifier)
            grad_model = keras.Model(
                inputs=classifier.inputs,
                outputs=[target_layer.output, classifier.output],
            )
            convolution_output, predictions = grad_model(
                {'image': image_batch}, training=False
            )
        else:
            backbone_index = classifier.layers.index(backbone)
            target_layer = find_last_conv_layer(backbone)
            backbone_grad = keras.Model(
                inputs=backbone.inputs,
                outputs=[target_layer.output, backbone.output],
            )
            backbone_input = _apply_layers(
                classifier.layers[:backbone_index], image_batch
            )
            convolution_output, predictions = backbone_grad(
                backbone_input, training=False
            )
            predictions = _apply_layers(
                classifier.layers[backbone_index + 1 :], predictions
            )
        if class_index is None:
            class_index = int(tf.argmax(predictions[0]))
        class_score = predictions[:, class_index]

    gradients = tape.gradient(class_score, convolution_output)
    if gradients is None:
        raise ValueError('Gradients were not produced for the selected layer.')

    weights = tf.reduce_mean(gradients, axis=(1, 2))
    feature_map = convolution_output[0]
    pooled_weights = weights[0]
    heatmap = tf.reduce_sum(feature_map * pooled_weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    maximum = tf.reduce_max(heatmap)
    heatmap = tf.where(maximum > 0, heatmap / maximum, tf.zeros_like(heatmap))
    return heatmap.numpy(), target_layer.name


def make_overlay(image, heatmap, alpha=0.45):
    original = np.asarray(image.convert('RGB').resize(IMG_SIZE), dtype='float32') / 255.0
    resized_heatmap = PILImage.fromarray(
        np.uint8(np.clip(heatmap, 0, 1) * 255)
    ).resize(IMG_SIZE, resample=PILImage.Resampling.BILINEAR)
    heatmap_array = np.asarray(resized_heatmap, dtype='float32') / 255.0
    colors = plt.get_cmap('jet')(np.uint8(heatmap_array * 255))[..., :3]
    return np.clip((1 - alpha) * original + alpha * colors, 0, 1)


target_layer = find_last_conv_layer(best_model)
print(f'Grad-CAM target layer: {target_layer.name}')



In [ ]:
number_of_examples = min(3, len(test_data))
example_rows = test_data.sample(number_of_examples, random_state=SEED)
fig, axes = plt.subplots(number_of_examples, 3, figsize=(12, 4 * number_of_examples), squeeze=False)

for row_index, (_, row) in enumerate(example_rows.iterrows()):
    image, image_batch = prepare_image(row['filepath'])
    probabilities = best_model(image_batch, training=False).numpy()[0]
    predicted_index = int(np.argmax(probabilities))
    heatmap, layer_name = make_gradcam_heatmap(
        image_batch, best_model, class_index=predicted_index, target_layer=target_layer
    )
    overlay = make_overlay(image, heatmap)
    predicted_name = DISPLAY_NAMES[CLASS_NAMES[predicted_index]]
    true_name = DISPLAY_NAMES[row['label']]

    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(f'True: {true_name}')
    axes[row_index, 1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
    axes[row_index, 1].set_title(f'Heatmap: {layer_name}')
    axes[row_index, 2].imshow(overlay)
    axes[row_index, 2].set_title(f'Predicted: {predicted_name} ({probabilities[predicted_index] * 100:.1f}%)')
    for axis in axes[row_index]:
        axis.axis('off')

fig.suptitle('Grad-CAM examples')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'gradcam_examples.png', dpi=160, bbox_inches='tight')
plt.show()

## 11. Export Model and Serving Metadata

The model is saved as a standalone `.keras` artifact. The JSON metadata records the preprocessing and class contract that the FastAPI service must follow.

For the application, configure `MODEL_PATH` to this exported model or copy it to the repository location expected by the backend.

In [ ]:
metadata_artifact = {
    'model_filename': DEPLOY_MODEL_PATH.name,
    'selected_model': selected_name,
    'architecture': ARCHITECTURE,
    'head_learning_rate': HEAD_LEARNING_RATE,
    'fine_tune_learning_rate': FINE_TUNE_LEARNING_RATE,
    'fine_tune_at_epoch': FINE_TUNE_AT_EPOCH,
    'fine_tune_layers': FINE_TUNE_LAYERS,
    'freeze_backbone_bn': True,
    'weight_decay': WEIGHT_DECAY,
    'label_smoothing': LABEL_SMOOTHING,
    'dropout_rate': DROPOUT_RATE,
    'input_size': list(IMG_SIZE),
    'channels': CHANNELS,
    'color_mode': 'RGB',
    'rescaling': 'pixel / 255.0',
    'class_names': CLASS_NAMES,
    'class_indices': EXPECTED_CLASS_INDICES,
    'seed': SEED,
    'split': {'train': 0.8, 'validation': 0.1, 'test': 0.1},
    'dataset_dir': str(DATASET_DIR.resolve()),
    'dataset_images': int(len(data)),
    'tensorflow_version': tf.__version__,
    'gpu_devices': GPU_NAMES,
    'replicas_in_sync': int(strategy.num_replicas_in_sync),
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'class_weight_cap': CLASS_WEIGHT_CAP,
    'epochs_requested': EPOCHS,
    'epochs_completed': len(combined_history['loss']),
    'class_weights': {str(index): weight for index, weight in CLASS_WEIGHTS.items()},
    'train_class_counts': {
        name: int(count) for name, count in train_class_counts.items()
    },
    'test_metrics': {
        'loss': selected_result['test_loss'],
        'accuracy': selected_result['test_accuracy'],
        'macro_f1': selected_result['macro_f1'],
        'mean_confidence': mean_confidence,
        'unique_predicted_classes': unique_predicted,
    },
    'tta_test_metrics': {
        'views': TTA_METRICS['views'],
        'accuracy': TTA_METRICS['test_accuracy'],
        'macro_f1': TTA_METRICS['test_macro_f1'],
    },
    'sanity_checks': SANITY_CHECKS,
    'layout_optimizer_disabled': True,
}

(OUTPUT_DIR / 'model_metadata.json').write_text(json.dumps(metadata_artifact, indent=2))
(OUTPUT_DIR / 'class_names.json').write_text(json.dumps(CLASS_NAMES, indent=2))
with (OUTPUT_DIR / 'model_summary.txt').open('w') as summary_file:
    best_model.summary(print_fn=lambda line: summary_file.write(line + '\n'))

print('Exported artifacts:')
for artifact in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {artifact.name}')


## Notes for Reuse

- Run the notebook on Kaggle with a GPU accelerator, or set `DATASET_DIR` and `OUTPUT_DIR` explicitly in another environment.
- Near-duplicate source images are grouped by perceptual hash (`dHash <= 10`) before splitting with `StratifiedGroupKFold` so duplicate families never straddle train, validation, and test splits.
- Test metrics describe this dataset split and should not be treated as guaranteed field performance or a definitive agricultural diagnosis.
- Class weights are computed from the training split only; validation and test metrics remain unweighted.
- Transfer-learning benchmarks were removed from this operational notebook. If they are still needed for a paper, keep them in a separate comparison notebook.